In [ ]:
# ============================================================
# KG1 v51 FINAL2 — SINGLE CELL
# Compativel: H100 / A100 / Blackwell
# BF16 + low_cpu_mem_usage (sem NF4, sem find_spec patch)
# ============================================================

!pip install -q peft datasets accelerate trl huggingface_hub safetensors pandas

import subprocess, sys, os, json, random, time, zipfile, shutil, re, math, types, gc
import importlib, importlib.machinery
from datetime import datetime, timezone
from collections import Counter

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ==================== STUBS ====================
class _Stub:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return a[0] if a else None
    def __getattr__(self, name): return _Stub()

for pkg in ['mamba_ssm', 'mamba_ssm.ops', 'mamba_ssm.ops.triton',
            'mamba_ssm.ops.triton.layernorm_gated',
            'mamba_ssm.ops.triton.selective_state_update',
            'mamba_ssm.ops.triton.ssd_combined',
            'mamba_ssm.utils', 'mamba_ssm.utils.generation',
            'causal_conv1d', 'causal_conv1d.causal_conv1d_interface']:
    if pkg not in sys.modules:
        m = types.ModuleType(pkg)
        m.__version__ = '0.0.0'
        m.__spec__ = importlib.machinery.ModuleSpec(pkg, None)
        m.__path__ = []
        m.__file__ = 'stub'
        for attr in ['RMSNormGated', 'rmsnorm_fn', 'selective_state_update',
                     'mamba_chunk_scan_combined', 'mamba_split_conv1d_scan_combined',
                     'InferenceParams', 'GenerationMixin',
                     'causal_conv1d_fn', 'causal_conv1d_update']:
            setattr(m, attr, _Stub)
        sys.modules[pkg] = m

ms = sys.modules['mamba_ssm']
ms.ops = sys.modules['mamba_ssm.ops']
ms.ops.triton = sys.modules['mamba_ssm.ops.triton']
ms.ops.triton.layernorm_gated = sys.modules['mamba_ssm.ops.triton.layernorm_gated']
ms.ops.triton.selective_state_update = sys.modules['mamba_ssm.ops.triton.selective_state_update']
ms.ops.triton.ssd_combined = sys.modules['mamba_ssm.ops.triton.ssd_combined']
ms.utils = sys.modules['mamba_ssm.utils']
ms.utils.generation = sys.modules['mamba_ssm.utils.generation']
cc = sys.modules['causal_conv1d']
cc.causal_conv1d_fn = _Stub()
cc.causal_conv1d_update = _Stub()
print('Stubs OK')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {vram_gb:.1f} GB')

try:
    from transformers.utils.import_utils import is_flash_attn_greater_or_equal_2_10
except ImportError:
    import transformers.utils.import_utils as _tiu
    _tiu.is_flash_attn_greater_or_equal_2_10 = lambda: False

import pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download

# ==================== AUTH ====================
def _get_secret(*names):
    try:
        from google.colab import userdata
        for n in names:
            v = userdata.get(n)
            if v: return v
    except Exception: pass
    for n in names:
        v = os.environ.get(n)
        if v: return v
    return ''

HF_TOKEN = _get_secret('HF_KEY', 'HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF login OK')

KAGGLE_USERNAME = _get_secret('KAGGLE_USERNAME') or 'felipe1983'
KAGGLE_KEY = _get_secret('KAGGLE_KEY')
if KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    kpath = os.path.expanduser('~/.kaggle/kaggle.json')
    with open(kpath, 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(kpath, 0o600)
    print(f'Kaggle: {KAGGLE_USERNAME}')

api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()

# ==================== CONFIG ====================
DATA_REPO = 'felipesp1983/kg1-nemotron-training'
OUTPUT_REPO = 'felipesp1983/kg1-nemotron-lora-v51-perfect'
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
COMPETITION = 'nvidia-nemotron-model-reasoning-challenge'
N_EXAMPLES = 5000
N_EPOCHS = 2
SUBMIT_STEPS = [200, 400, 600, 800, 1000, 1200]
CONFIG = {
    'lora_rank': 32, 'lora_alpha': 16, 'lora_dropout': 0.05,
    'target_modules': 'all-linear', 'learning_rate': 5e-5,
    'per_device_batch_size': 1, 'gradient_accumulation_steps': 8,
    'max_length': 1024, 'warmup_ratio': 0.05, 'weight_decay': 0.01,
    'lr_scheduler': 'cosine', 'optim': 'adamw_torch',
    'output_dir': '/tmp/kg1_output/v51',
}

try:
    for p in ['/usr/local/cuda-12.8/bin/ptxas', '/usr/local/cuda/bin/ptxas']:
        if os.path.exists(p):
            t = os.path.join(os.path.dirname(shutil.which('python') or '/usr/bin/python'), 'ptxas')
            if not os.path.exists(t): shutil.copy2(p, t)
            break
except Exception: pass

print(f'Config: {N_EXAMPLES}ex, {N_EPOCHS}ep, LR={CONFIG["learning_rate"]}, r={CONFIG["lora_rank"]}, a={CONFIG["lora_alpha"]}')

# ==================== LOAD DATA ====================
print('\n=== Loading data ===')
all_examples = []
try:
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/sft_v51_perfect.jsonl', local_dir='/tmp/kg1_data')
    with open('/tmp/kg1_data/data/sft_v51_perfect.jsonl') as f:
        for line in f:
            all_examples.append(json.loads(line))
    print(f'Loaded: {len(all_examples)} solver-enhanced examples')
except Exception as e:
    print(f'v51 data failed ({e}), using train.csv')
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/train.csv', local_dir='/tmp/kg1_data')
    for _, row in pd.read_csv('/tmp/kg1_data/data/train.csv').iterrows():
        all_examples.append({
            'prompt': row['prompt'] + '\nPut your final answer inside \\boxed{}.',
            'completion': '\\boxed{' + str(row['answer']) + '}',
            'family': 'unknown'})
    print(f'Loaded: {len(all_examples)} raw examples')

def classify(t):
    p = t.lower()
    if 'bit manipulation' in p: return 'bit'
    if 'gravitational' in p: return 'grav'
    if 'unit conversion' in p or 'measurement' in p: return 'unit'
    if 'numeral' in p: return 'num'
    if 'encryption' in p: return 'enc'
    if 'transformation' in p: return 'eq'
    return 'other'

random.seed(42)
by_family = {}
for ex in all_examples:
    fam = ex.get('family') or classify(ex.get('prompt', ''))
    by_family.setdefault(fam, []).append(ex)

shares = {'grav':1, 'unit':1, 'num':1, 'enc':1, 'cipher':1, 'bit':1.5,
          'eq':2.5, 'equation':2.5, 'gravity':1, 'numeral':1}
total_shares = sum(shares.get(f, 1.0) for f in by_family)
base_n = N_EXAMPLES / total_shares
examples = []
for fam, pool in by_family.items():
    n = int(base_n * shares.get(fam, 1.0))
    if n <= len(pool):
        examples.extend(random.sample(pool, n))
    else:
        examples.extend(pool + random.choices(pool, k=n - len(pool)))
random.shuffle(examples)
examples = examples[:N_EXAMPLES]

formatted = []
for ex in examples:
    formatted.append({'messages': [
        {'role': 'user', 'content': ex.get('prompt', '')},
        {'role': 'assistant', 'content': ex.get('completion', '')}]})

fam_counts = Counter(classify(e['messages'][0]['content']) for e in formatted)
print(f'Dataset: {len(formatted)} examples')
for fam, cnt in sorted(fam_counts.items()):
    print(f'  {fam}: {cnt} ({cnt/len(formatted)*100:.1f}%)')

# ==================== DOWNLOAD MODEL ====================
print('\n=== Downloading model ===')
model_path = '/tmp/nemotron_model'
safetensor_count = 0
if os.path.exists(model_path):
    safetensor_count = len([f for f in os.listdir(model_path)
                           if f.endswith('.safetensors') and 'index' not in f])

if safetensor_count < 13:
    os.makedirs(model_path, exist_ok=True)
    BASE_URL = 'https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16/resolve/main'
    meta = ['config.json', 'configuration_nemotron_h.py', 'modeling_nemotron_h.py',
            'generation_config.json', 'tokenizer_config.json', 'tokenizer.json',
            'special_tokens_map.json', 'chat_template.jinja', 'model.safetensors.index.json']
    shards = [f'model-{str(i).zfill(5)}-of-00013.safetensors' for i in range(1, 14)]
    for f in meta + shards:
        out = os.path.join(model_path, f)
        min_sz = 1_000_000_000 if f.startswith('model-') else 100
        if os.path.exists(out) and os.path.getsize(out) > min_sz:
            continue
        print(f'  {f}...')
        for attempt in range(3):
            r = subprocess.run(['curl', '-L', '-C', '-', '--retry', '3', '--retry-delay', '5',
                                '-o', out, f'{BASE_URL}/{f}'], capture_output=True, text=True)
            if r.returncode == 0 and os.path.exists(out) and os.path.getsize(out) > min_sz:
                print(f'    OK ({os.path.getsize(out)/1e9:.1f}GB)')
                break
            print(f'    retry {attempt+1}/3')
    total_sz = sum(os.path.getsize(os.path.join(model_path, f))
                   for f in os.listdir(model_path)
                   if f.endswith('.safetensors') and 'index' not in f)
    print(f'Model: {total_sz/1e9:.1f}GB')
else:
    print(f'Model cached ({safetensor_count} shards)')

# ==================== LOAD MODEL ====================
print('\n=== Loading model ===')
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from peft import LoraConfig, get_peft_model

_gpu_cap = float(f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}')
IS_BLACKWELL = _gpu_cap >= 10.0

_model_kwargs = dict(
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

if IS_BLACKWELL:
    _cfg = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
    if hasattr(_cfg, 'use_mamba_kernels'):
        _cfg.use_mamba_kernels = False
        _model_kwargs['config'] = _cfg
        print('[BLACKWELL] Mamba kernels DISABLED')

model = AutoModelForCausalLM.from_pretrained(model_path, **_model_kwargs)

for module in model.modules():
    if hasattr(module, 'is_fast_path_available'):
        module.is_fast_path_available = False

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'Model: {model.num_parameters()/1e9:.1f}B params, VRAM: {vram_used:.1f}GB')

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer: vocab={tokenizer.vocab_size}')

# ==================== LORA ====================
print(f'\n=== LoRA r={CONFIG["lora_rank"]} alpha={CONFIG["lora_alpha"]} ===')
model.enable_input_require_grads()
lora_config = LoraConfig(
    r=CONFIG['lora_rank'], lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'], target_modules=CONFIG['target_modules'],
    bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.1f}GB')

# ==================== DATASET ====================
print('\n=== Preparing dataset ===')
from datasets import Dataset
texts = [tokenizer.apply_chat_template(ex['messages'], tokenize=False,
         add_generation_prompt=False) for ex in formatted]
ds = Dataset.from_dict({'text': texts})
sample_lens = [len(tokenizer(t)['input_ids']) for t in texts[:100]]
print(f'Dataset: {len(ds)} examples')
print(f'Tokens: min={min(sample_lens)} max={max(sample_lens)} mean={sum(sample_lens)/len(sample_lens):.0f}')

# ==================== TRAIN ====================
print('\n=== Training ===')
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
os.makedirs(CONFIG['output_dir'], exist_ok=True)

training_args = SFTConfig(
    output_dir=CONFIG['output_dir'],
    dataset_text_field='text',
    max_length=CONFIG['max_length'],
    packing=False,
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    lr_scheduler_type=CONFIG['lr_scheduler'],
    optim=CONFIG['optim'],
    bf16=True,
    logging_steps=5,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=15,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    dataloader_num_workers=0,
    max_grad_norm=1.0,
)

class UploadCallback(TrainerCallback):
    def __init__(self, repo_id, submit_steps):
        self.repo_id = repo_id
        self.submit_steps = set(submit_steps)
        self.submitted = set()
        self.hf_api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
        try:
            self.hf_api.create_repo(repo_id, private=True, exist_ok=True)
        except Exception:
            pass

    def on_save(self, args, state, control, **kwargs):
        import glob as g
        step = state.global_step
        loss_val = 'N/A'
        if state.log_history:
            for entry in reversed(state.log_history):
                if 'loss' in entry:
                    loss_val = entry['loss']
                    break
        ckpts = sorted(g.glob(f'{args.output_dir}/checkpoint-*'))
        if not ckpts:
            return
        try:
            self.hf_api.upload_folder(
                folder_path=ckpts[-1],
                path_in_repo=f'checkpoint-{step}',
                repo_id=self.repo_id,
                commit_message=f'Step {step} | Loss {loss_val}')
            print(f'\n>>> HF upload OK: step {step}, loss={loss_val}')
        except Exception as e:
            print(f'\n>>> HF upload FAILED: {e}')

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        loss = logs.get('loss', 0)
        if isinstance(loss, float) and (math.isnan(loss) or math.isinf(loss)):
            print(f'\n!!! NaN/Inf at step {state.global_step}')
            control.should_training_stop = True
            return
        if isinstance(loss, (int, float)) and loss > 30.0 and state.global_step > 5:
            print(f'\n!!! Loss explosion {loss:.2f} at step {state.global_step}')
            control.should_training_stop = True

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[UploadCallback(OUTPUT_REPO, SUBMIT_STEPS)],
)

total_steps = (len(ds) // CONFIG['gradient_accumulation_steps']) * N_EPOCHS
print(f'Steps: ~{total_steps}, Est: ~{total_steps * 55 / 3600:.1f}h')

start = time.time()
try:
    trainer.train()
except Exception as e:
    print(f'\n!!! Training error: {e}')
    try:
        model.save_pretrained(CONFIG['output_dir'])
        tokenizer.save_pretrained(CONFIG['output_dir'])
    except Exception:
        pass

elapsed = time.time() - start
print(f'\nTraining: {elapsed/3600:.2f}h')

# ==================== SAVE + UPLOAD ====================
print('\n=== Saving ===')
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])

final_loss = 'N/A'
if trainer.state.log_history:
    for entry in reversed(trainer.state.log_history):
        if 'loss' in entry:
            final_loss = entry['loss']
            break

try:
    api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=CONFIG['output_dir'],
        repo_id=OUTPUT_REPO,
        commit_message=f'v51 FINAL: {len(formatted)}ex, {N_EPOCHS}ep, loss={final_loss}')
    print(f'Uploaded to: {OUTPUT_REPO}')
except Exception as e:
    print(f'Upload failed: {e}')

# ==================== SMART STRIP + SUBMIT ====================
print('\n=== Smart Strip ===')
import glob
from safetensors.torch import load_file, save_file

ckpt_dir = f'{CONFIG["output_dir"]}/checkpoint-400'
if not os.path.exists(ckpt_dir):
    ckpts = sorted(glob.glob(f'{CONFIG["output_dir"]}/checkpoint-*'),
                   key=lambda x: int(x.split('-')[-1]))
    ckpt_dir = ckpts[-1] if ckpts else CONFIG['output_dir']
    print(f'Using: {ckpt_dir}')

sf_path = os.path.join(ckpt_dir, 'adapter_model.safetensors')
cfg_path = os.path.join(ckpt_dir, 'adapter_config.json')
tensors = load_file(sf_path)
routed_re = re.compile(r'\.experts\.\d+\.')
keep = {k: v for k, v in tensors.items() if not routed_re.search(k)}
print(f'Kept: {len(keep)} | Removed routed: {len(tensors) - len(keep)}')

out_dir = '/tmp/kg1_submit/stripped'
os.makedirs(out_dir, exist_ok=True)
save_file(keep, os.path.join(out_dir, 'adapter_model.safetensors'))

with open(cfg_path) as f:
    cfg = json.load(f)
kept_mods = set()
for k in keep:
    for m in ['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj','gate']:
        if m in k:
            kept_mods.add(m)
cfg['target_modules'] = sorted(kept_mods)
if 'base_model_name_or_path' in cfg:
    cfg['base_model_name_or_path'] = MODEL_NAME
with open(os.path.join(out_dir, 'adapter_config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)

zip_path = '/tmp/kg1_submit/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(os.path.join(out_dir, fname), fname)
print(f'ZIP: {os.path.getsize(zip_path)/1e6:.1f} MB')

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    assert len(names) == 2 and all('/' not in n for n in names)
print(f'ZIP OK: {names}')

step_str = ckpt_dir.split('-')[-1] if 'checkpoint' in ckpt_dir else 'final'
desc = f'v51-step{step_str}-smart-strip'
print(f'\nSubmitting: {desc}')
os.system(f'kaggle competitions submit -c {COMPETITION} -f {zip_path} -m "{desc}"')

print(f'\n{"="*60}')
print(f'  v51 COMPLETE')
print(f'  GPU: {torch.cuda.get_device_name(0)}')
print(f'  Examples: {len(formatted)} | Epochs: {N_EPOCHS}')
print(f'  Final loss: {final_loss}')
print(f'  Time: {elapsed/3600:.2f}h')
print(f'  Submitted: {desc}')
print(f'{"="*60}')
